In [ ]:
from cellpose import models
from glob import glob
from natsort import natsorted
import tifffile
import matplotlib.pyplot as plt
from pathlib import Path

# Define paths
image_dir = "/pscratch/sd/x/xchong/sam3_finetune/seg_annotation_pipeline2/data/images"
model_path = "/pscratch/sd/x/xchong/3RSE/832/cellpose/models/petiole_model_flow0"  # Update this

# Load all TIFF files
print("Loading images...")
test_files = natsorted(glob(f"{image_dir}/*.tif") + glob(f"{image_dir}/*.tiff"))
test_files = [f for f in test_files if "_masks" not in f and "_flows" not in f]
test_data = [tifffile.imread(f) for f in test_files]

print(f"Loaded {len(test_data)} images")

# Load model and run inference
print("Running inference...")
model = models.CellposeModel(gpu=True, pretrained_model=model_path)
masks = model.eval(test_data, batch_size=32)[0]

print(f"Inference complete! Detected cells in each image:")
for i, (file, mask) in enumerate(zip(test_files, masks)):
    print(f"  {Path(file).name}: {mask.max()} cells")

# Plot first few results
n_show = min(6, len(test_data))
fig, axes = plt.subplots(2, n_show, figsize=(3*n_show, 6))

for i in range(n_show):
    axes[0, i].imshow(test_data[i], cmap='gray')
    axes[0, i].set_title(Path(test_files[i]).name)
    axes[0, i].axis('off')
    
    axes[1, i].imshow(masks[i], cmap='jet')
    axes[1, i].set_title(f'{masks[i].max()} cells')
    axes[1, i].axis('off')

plt.tight_layout()
plt.savefig('inference_results.png', dpi=150, bbox_inches='tight')
plt.show()

Loading images...
Loaded 42 images
Running inference...


In [1]:
from cellpose import models
import tifffile
import matplotlib.pyplot as plt
from pathlib import Path

# Define paths
image_path = "/pscratch/sd/x/xchong/sam3_finetune/seg_annotation_pipeline2/data/images/20260212_133951_petiole30_00100.tiff"
model_path = "/pscratch/sd/x/xchong/3RSE/832/cellpose/models/petiole_model_flow0"


# Load image
print("Loading image...")
img = tifffile.imread(image_path)
print(f"Loaded {Path(image_path).name}")

# Load model and run inference
print("Running inference...")
model = models.CellposeModel(gpu=True, pretrained_model=model_path)

mask = model.eval([img], batch_size=1)[0][0]

print(f"Inference complete!")
print(f"{Path(image_path).name}: {mask.max()} cells")

# Save mask
tifffile.imwrite("20260212_133951_petiole30_00100_mask.tiff", mask.astype("uint16"))

# Plot result
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(img, cmap="gray")
axes[0].set_title(Path(image_path).name)
axes[0].axis("off")

axes[1].imshow(mask, cmap="jet")
axes[1].set_title(f"{mask.max()} cells")
axes[1].axis("off")

plt.tight_layout()
plt.savefig("20260212_133951_petiole30_00100_result.png", dpi=150, bbox_inches="tight")
plt.show()

Loading image...
Loaded 20260212_133951_petiole30_00100.tiff
Running inference...


RuntimeError: CUDA error: CUBLAS_STATUS_INVALID_VALUE when calling `cublasGemmStridedBatchedEx(handle, opa, opb, (int)m, (int)n, (int)k, (void*)&falpha, a, CUDA_R_16BF, (int)lda, stridea, b, CUDA_R_16BF, (int)ldb, strideb, (void*)&fbeta, c, std::is_same_v<C_Dtype, float> ? CUDA_R_32F : CUDA_R_16BF, (int)ldc, stridec, (int)num_batches, compute_type, CUBLAS_GEMM_DEFAULT_TENSOR_OP)`